<a href="https://colab.research.google.com/github/Saransh-Basu-01/50-Ml-Projects-to-understand-LLMs/blob/main/Project_20_Perplexity_over_Sequences_texts_and_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import requests
from datasets import load_dataset

from tqdm import tqdm

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer

# Part_1:Perplexity over Time

## Task_1:
- Import the GPT-2 model and its tokenizer. Switch the model into
eval mode and push it to the GPU. You’ll import several models
later in the project, and use the variable name model in a function
in Part 2, so I recommend naming the model here gpt2.
Import the text for the book Frankenstein’s Monster from guten
berg.org. You can find and copy code from Chapter 2, or check the
online solution for the importing code.
Define a variable for the sequence length to use in averaging
the loss as the maximum sequence length of the model (hint: check
gpt2.config), and calculate the number of non-overlapping text
segments that can be extracted from the text tokens.

In [2]:
gpt2tokenizer=AutoTokenizer.from_pretrained('gpt2')
gpt2=AutoModelForCausalLM.from_pretrained('gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
gpt2.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpt2.to(device);


In [5]:
device

device(type='cuda')

In [14]:
baseurl = 'https://www.gutenberg.org/cache/epub/84/pg84.txt'
text = requests.get(baseurl).text
print(text[:1000])


The Project Gutenberg eBook of Frankenstein; or, The modern Prometheus
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Frankenstein; or, The modern Prometheus

Author: Mary Wollstonecraft Shelley


        
Release date: October 1, 1993 [eBook #84]
                Most recently updated: February 10, 2026

Language: English

Other information and formats: www.gutenberg.org/ebooks/84

Credits: Judith Boss, Christy Phillips, Lynn Hanninen and David Meltzer. HTML version by Al Haines.
        Further corrections by Menno de Leeuw.


*** START OF THE PROJECT GUTEN


In [7]:
tokens =gpt2tokenizer.encode(text,return_tensors='pt')
num_tokens =tokens.shape[-1]
print(f'Number of tokens: {num_tokens:,}')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (114211 > 1024). Running this sequence through the model will result in indexing errors


Number of tokens: 114,211


In [8]:
gpt2.config

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.20.2",
  "use_cache": true,
  "vocab_size": 50257
}

In [9]:
len(tokens[0]),tokens.shape[-1]

(114211, 114211)

In [10]:
# max sequence length
seq_len =gpt2.config.n_positions
print(f'Sequence length: {seq_len}')

# how many samples fit into the data
nSegments =len(tokens[0])//seq_len
print(f'Number of non-overlapping segments: {nSegments:,}')

Sequence length: 1024
Number of non-overlapping segments: 111


## Task_2
- Create a for-loop over segments, in which the following steps are
implemented.
1. Define start and endpoint segment indices into the token se
quence. Each segment is 1024 tokens long, and non-overlapping
with the previous segment. (Hint: The start index can be de
fined as a multiple of the looping index variable.)
2. Extract the tokens from those indices and move them to the
GPU.
3. Run a forward pass and get the outputs and the loss. You
can implement that on your own as you learned in Project 19,
although it’s easiest to use the labels= optional input argument
when calling the model.
4. Accumulate the segment-averaged loss. This is the summation
term inside the natural exponential in Equation 4.13. You only
need to sum here; do not apply the natural exponential.
5. Implement the full Equation 4.13 on the loss from this segment,
and store as an element in an array to be used for plotting.
6. After the for-loop, finalize calculating the total perplexity from
Step 4 by implementing the rest of Equation 4.13.
Visualize the per-segment perplexity values, the perplexity of the ac
cumulated losses (from Step 6), and the average of the per-segment
perplexities (from Step 5), as in Figure 4.16

In [11]:
tokens[0][0:1024].unsqueeze(0).shape

torch.Size([1, 1024])

In [12]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [13]:
perplexities = np.zeros(nSegments)

sum_losses = 0.

for segi in tqdm(range(nSegments)):
  print(tokens[0].max(), gpt2.config.vocab_size)
  # 1) start and end indices
  start = segi * seq_len
  end = start + seq_len

  # 2) extract the data and push to the GPU
  X = tokens[0][start:end].unsqueeze(0).to(device)

  # 3) forward pass
  with torch.no_grad():
    outputs = gpt2(X,labels=X)

  # 4) accumulated loss for later perplexity calculations
  sum_losses += outputs.loss.item()

  # 5) per-segment perplexity
  perplexities[segi] = torch.exp(outputs.loss).item()

# 6) calculate perplexity
ave_perplexity = torch.exp( torch.tensor(sum_losses) / nSegments )

  0%|          | 0/111 [00:00<?, ?it/s]

tensor(50255) 50257


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
  2%|▏         | 2/111 [00:01<01:00,  1.81it/s]

tensor(50255) 50257
tensor(50255) 50257


  4%|▎         | 4/111 [00:01<00:28,  3.80it/s]

...

 95%|█████████▌| 106/111 [00:09<00:00, 12.34it/s]

tensor(50255) 50257
tensor(50255) 50257
tensor(50255) 50257


 99%|█████████▉| 110/111 [00:10<00:00, 12.29it/s]

tensor(50255) 50257
tensor(50255) 50257
tensor(50255) 50257


100%|██████████| 111/111 [00:10<00:00, 10.94it/s]
